# Possession pipeline: train on external games, test, and write a submission

**Upgraded 5-Layer Pipeline** (`possession/` package)

1. **Broadcast-to-Tactical Alignment & Ball Fusion (`possession/broadcast.py`)** - aligns live broadcast camera shots (`valid == 1`) to the tactical timeline using permutation-invariant team constellation matching and fills missing tactical ball frames with broadcast ball observations.
2. **Bidirectional Ball Bridging (`possession/ball.py`)** - cubic-Hermite curve between ball sightings on both sides of remaining gaps.
3. **Collective Kinematics & Pass-Bridge Models (`possession/features.py`, `possession/models.py`)** - LightGBM dead-ball and control models using:
   - **Collective team motion** (`02_tracking_atd.ipynb`): team centroid speed, movement coherence, spatial spread, and radial convergence toward the ball.
   - **Deflection-filtered Pass Bridge** (`CHALLENGE.md`): suppresses isolated brief touches (< 0.4 s) sandwiched between opponent touches and flags completed same-team pass intervals (`o_pass_bridge`).
4. **Smart Tagging Set-Piece Anchors & Tactical Phase Priors (`possession/smart_tagging.py`)** - uses set-piece windows (`01_smart_tagging.ipynb`) + ball acceleration to pinpoint exact restart frames, and maps offensive (`BUILD UP`, `PROGRESSION`, ...) vs. defensive (`DEFENDING IN ...`, `DEFENSIVE TRANSITION`) tags to possessing-team log-odds priors.
5. **Sticky Viterbi Decoder (`possession/decode.py`)** - global sequence decoding over `{H, A, D}` with short-run clean-up, tuned for **`spellF1`** on the external games.

**Rules this notebook respects** (from the organisers' clarification email)
- Training and tuning use **only the external Metrica games** in `external_data/`.
- The challenge match is only ever *predicted on*. The public 5-minute ground truth is used **once, at the very end, as a pipeline sanity check** - never to choose parameters.
- Do not paste provided-match data (tracking, tagging, video, ground truth) into any AI service.

In [ ]:
%load_ext autoreload
%autoreload 2
import os, sys, time, json, subprocess
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

ROOT = os.getcwd()
while not os.path.isfile(os.path.join(ROOT, "games.py")):
    parent = os.path.dirname(ROOT)
    assert parent != ROOT, "run this notebook from inside the repository"
    ROOT = parent
sys.path.insert(0, ROOT)
os.chdir(ROOT)

from possession.data import load_external_games, load_challenge
from possession.labels import build_labels, label_report, H, A, D, X
from possession.masking import MaskConfig, apply_masks, missing_stats
from possession.ball import fill_ball, gap_error_table
from possession.models import prepare, PossessionModels, frame_metrics
from possession.decode import DecodeConfig, decode_match, tune_decoder, score, nearest_player_baseline
from possession.smart_tagging import simulate_external_smart_tagging, apply_smart_tagging_priors
from possession.pipeline import predict_challenge, write_submission

# ---- configuration ------------------------------------------------------------------------
TEAM_NAME   = "myteam"          # -> evaluation/results/<TEAM_NAME>.csv
MASK_CFGS = {"sim": MaskConfig(mode="simulate"),   # ball hidden independently of the label (live-play gaps)
             "raw": MaskConfig(mode="raw")}        # the external tracker's own gaps (long, mostly dead ball)
TRAIN_RUNS  = [("sim", 1), ("sim", 2), ("raw", 3), ("raw", 4)]   # (mask mode, seed) per training game
TEST_SEED   = 100                # mask seed used for out-of-fold evaluation
STRIDE_TRAIN = 4                 # keep every 4th frame for training (frames are highly correlated)
N_ROUNDS    = 350
USE_BROADCAST = True             # fuse broadcast ball into missing tactical ball frames
USE_SMART_TAGGING = True         # apply Smart Tagging restart anchors & phase priors
print("root:", ROOT)

## 1. External games and event-derived labels

In [ ]:
games = load_external_games()
pd.DataFrame([g.summary() for g in games])

In [ ]:
labels = {g.name: build_labels(g) for g in games}
pd.DataFrame({name: label_report(c) for name, c in labels.items()}).T

**Read this table before trusting any score.** The challenge ground truth has ~17 possessions per 5 minutes with mean ~14 s.

**Important data quirk handled by the dual masking modes:** in the raw external games the ball is missing mostly *while the ball is dead*. Training on both `sim` (live-play ball gaps) and `raw` (tracker's natural long dead-ball gaps) teaches the models to handle both regimes.

In [ ]:
rows = []
for g in games:
    c = labels[g.name]; nan = np.isnan(g.ball[:, 0])
    rows.append({"game": g.name, "ball NaN | dead %": round(100*nan[c==D].mean(),1),
                 "ball NaN | live %": round(100*nan[(c==H)|(c==A)].mean(),1)})
pd.DataFrame(rows)

## 2. Simulating the challenge's missing ball (Layer 1 diagnostics)

In [ ]:
g = games[0]
mm = apply_masks(g, MASK_CFGS["sim"], TEST_SEED)
print(missing_stats(mm))
filled, obs = fill_ball(mm.ball)
display(gap_error_table(g.ball_true, filled, obs))

lo = int(np.flatnonzero(g.period > 0)[0]) + 25*60*10; hi = lo + 25*60*2
fig, ax = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
t = np.arange(lo, hi) / 25.0
ax[0].plot(t, filled[lo:hi, 0], lw=1, label="bridged ball x"); ax[0].plot(t, g.ball_true[lo:hi, 0], lw=1, alpha=.6, label="true ball x")
ax[0].legend(); ax[0].set_ylabel("x (m)")
ax[1].fill_between(t, 0, (~obs[lo:hi]).astype(int), step="mid", color="tab:red", alpha=.5); ax[1].set_ylabel("ball masked"); ax[1].set_xlabel("s")
plt.tight_layout()

## 3. Build upgraded features (collective motion coherence + pass bridge + deflection filter)

In [ ]:
t0 = time.time()
train_bundles = {g.name: [prepare(g, MASK_CFGS[m], seed=s, stride=STRIDE_TRAIN) for m, s in TRAIN_RUNS] for g in games}
test_bundles  = {m: {g.name: prepare(g, MASK_CFGS[m], seed=TEST_SEED, stride=1) for g in games} for m in MASK_CFGS}
print("features built in %.0fs" % (time.time() - t0))
b = test_bundles["sim"][games[0].name]
print("dead features:", len(b.names_c), " control features:", len(b.names_o))

## 4. Leave-one-game-out training -> out-of-fold probabilities
Each game is predicted by models that never saw it.

In [ ]:
oof = []     # one entry per (held-out game, mask mode)
gmap = {g.name: g for g in games}
sim_tags = {g.name: simulate_external_smart_tagging(g, seed=42) for g in games}

for g in games:
    t0 = time.time()
    tr = [bd for h in games if h.name != g.name for bd in train_bundles[h.name]]
    m = PossessionModels.fit(tr, n_rounds=N_ROUNDS)
    for mode in MASK_CFGS:
        tb = test_bundles[mode][g.name]
        p_dead, p_home = m.predict(tb)
        mm = apply_masks(g, MASK_CFGS[mode], TEST_SEED)
        p_dead_st, p_home_st, _ = apply_smart_tagging_priors(mm, sim_tags[g.name], p_dead, p_home)
        print(g.name, mode, {k: round(v, 3) for k, v in frame_metrics(tb, p_dead, p_home).items()}, "%.0fs" % (time.time() - t0))
        oof.append(dict(name=g.name, mode=mode, p_dead=p_dead, p_home=p_home,
                        p_dead_st=p_dead_st, p_home_st=p_home_st,
                        period=tb.period, codes=tb.codes))

## 5. What does each layer buy? (out-of-fold, masked external games, official `spellF1` / `macroF1`)
- **nearest-player baseline** - the published baseline's logic (smoothing window 50) on the same masked data
- **models, argmax** - the two upgraded models with no sequence decoding
- **models + default decoder** - Viterbi with untuned costs
- **models + Smart Tagging priors + decoder** - adding set-piece restart pinpointing and phase priors

In [ ]:
def argmax_codes(o):
    out = np.where(o["p_dead"] > 0.5, D, np.where(o["p_home"] > 0.5, H, A)).astype(np.int8)
    out[o["period"] == 0] = X
    return out

rows = []
for o in oof:
    g = gmap[o["name"]]
    mm = apply_masks(g, MASK_CFGS[o["mode"]], TEST_SEED)
    for name, pred in (
        ("1. nearest baseline (w=50)", nearest_player_baseline(mm, 50)),
        ("2. models, argmax", argmax_codes(o)),
        ("3. models + decoder", decode_match(o["p_dead"], o["p_home"], o["period"], DecodeConfig(stride=5))),
        ("4. models + Smart Tagging + decoder", decode_match(o["p_dead_st"], o["p_home_st"], o["period"], DecodeConfig(stride=5))),
    ):
        s = score(o["codes"], pred)
        rows.append({"game": g.name, "mode": o["mode"], "method": name, **{k: round(v, 4) if isinstance(v, float) else v for k, v in s.items()}})
res = pd.DataFrame(rows)
display(res.pivot_table(index=["mode", "method"], columns="game", values="spell_f1"))
display(res.groupby(["mode", "method"])[["spell_f1", "macro_f1", "n_pred", "n_gt"]].mean().round(4))

## 6. Tune the decoder on the out-of-fold external predictions

In [ ]:
t0 = time.time()
oof_for_tune = [
    dict(name=o["name"], mode=o["mode"],
         p_dead=o["p_dead_st"] if USE_SMART_TAGGING else o["p_dead"],
         p_home=o["p_home_st"] if USE_SMART_TAGGING else o["p_home"],
         period=o["period"], codes=o["codes"])
    for o in oof
]
best_cfg, table = tune_decoder(oof_for_tune, grid_ld=(6, 12, 25, 50, 100))
print("tuned in %.0fs" % (time.time() - t0)); print(best_cfg)
display(table.sort_values("spell_f1", ascending=False).head(10).round(4))

In [ ]:
rows = []
for o in oof_for_tune:
    pred = decode_match(o["p_dead"], o["p_home"], o["period"], best_cfg)
    rows.append({"game": o["name"], "mode": o["mode"], **{k: round(v, 4) for k, v in score(o["codes"], pred).items()}})
pd.DataFrame(rows)

## 7. Fit the final models on all three external games and save everything

In [ ]:
all_bundles = [bd for g in games for bd in train_bundles[g.name]]
final = PossessionModels.fit(all_bundles, n_rounds=N_ROUNDS)
os.makedirs("artifacts/models", exist_ok=True)
final.save("artifacts/models")
with open("artifacts/models/decode_cfg.json", "w") as f:
    json.dump(best_cfg.__dict__, f, indent=2)
display(final.importance("ctrl", 15).round(0).to_frame("gain")); display(final.importance("dead", 12).round(0).to_frame("gain"))

## 8. Predict the challenge match (with Broadcast Fusion + Smart Tagging Priors)
Loads the tactical ATD, aligns live broadcast camera shots to fill missing tactical ball positions, computes the upgraded collective & pass-bridge features, applies Smart Tagging set-piece restart anchors and phase priors, and runs the Viterbi decoder.

In [ ]:
ch = load_challenge()
graded = ch.period > 0
ball_nan = np.isnan(ch.ball[:, 0])
print("graded frames:", int(graded.sum()), " tactical ball missing in graded: %.1f%%" % (100 * ball_nan[graded].mean()))
runs = np.diff(np.flatnonzero(np.diff(np.concatenate(([0], ball_nan[graded].astype(int), [0]))) != 0))[::2]
print("ball-missing gaps: n=%d  median %.1fs  p90 %.1fs  max %.1fs" % (len(runs), np.median(runs)/25, np.percentile(runs, 90)/25, runs.max()/25))
print("mean tracked slots in graded frames: %.1f" % (~np.isnan(ch.xy[graded, :, 0])).sum(1).mean())

In [ ]:
final = PossessionModels.load("artifacts/models")
best_cfg = DecodeConfig(**json.load(open("artifacts/models/decode_cfg.json")))
rule_w = json.load(open("artifacts/models/rules_cfg.json")) if os.path.isfile("artifacts/models/rules_cfg.json") else None
codes, p_dead, p_home, diag, variants = predict_challenge(
    final, best_cfg, ch,
    use_broadcast=USE_BROADCAST,
    use_smart_tagging=USE_SMART_TAGGING,
    rule_weights=rule_w,
)
print("pipeline diagnostics:", json.dumps(diag, indent=2))
path = write_submission(codes, os.path.join("evaluation", "results", TEAM_NAME + ".csv"))
g_ = codes[ch.period > 0]
print("wrote", path, " rows:", len(codes), " H/A/D share of graded: %.2f / %.2f / %.2f" % tuple((g_ == k).mean() for k in (H, A, D)))
print(subprocess.run([sys.executable, "evaluation/validate_submission.py", path], capture_output=True, text=True).stdout)

## 9. Pipeline sanity check on the public 5 minutes (NOT for tuning)
This only confirms frames are aligned and the file scores. **Do not use it to compare variants or change parameters** - that is explicitly forbidden, and 17 possessions cannot rank anything anyway.

In [ ]:
out = subprocess.run([sys.executable, "evaluation/run_evaluation.py"], capture_output=True, text=True)
print(out.stdout[:2200] or out.stderr[-2500:])   # the board is at the top; the tail is just the last baseline report